In [10]:
from IPython.display import HTML

# Observable
obs = [
   "#4269D0FF", "#F0BD3CFF", "#FF5D45FF", "#6CC5B0FF", "#3CA951FF", "#FF8AB7FF",
   "#A463F2FF", "#97BBF5FF", "#9C6B4EFF", "#9498A0FF", "#1B1B1BFF"
]

html = "".join(
    f'<div style="display:inline-block;width:100px;height:30px;background:{c};margin:2px">{c}</div>'
    for c in obs
)

HTML(html)

def parse_qmd_sections(file_path):
    sections = {}
    current_section = None
    current_content = []

    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            line_stripped = line.strip()

            # Check for H2 header
            if line_stripped.startswith("## "):
                # Save previous section
                if current_section is not None:
                    sections[current_section] = "".join(current_content).strip()

                # Start new section
                current_section = line_stripped[3:].strip()
                current_content = []
            else:
                if current_section is not None:
                    current_content.append(line)

        # Save last section
        if current_section is not None:
            sections[current_section] = "".join(current_content).strip()

    return sections

In [32]:
import numpy as np
import pandas as pd

INPUT_CSV = "../data/books_nonfiction.csv"

df = pd.read_csv(INPUT_CSV)
df = df.sort_values(by="Year", ascending=False)

if INPUT_CSV == "../data/books_fiction.csv":
    OUTPUT_QMD = "books_fiction.qmd"
    TEMPLATE_FILE = "template_books_fiction.qmd"
    print_tags = True
    print_comments = True
    print_country = True
    map_ratings = True
    external_comments = dict()
    print_commentless_as_wip = False
    print_subtitle = False

if INPUT_CSV == "../data/books_nonfiction.csv":
    OUTPUT_QMD = "books_nonfiction.qmd"
    TEMPLATE_FILE = "template_books_nonfiction.qmd"
    print_tags = True
    print_comments = True
    print_country = False
    map_ratings = False
    external_comments = parse_qmd_sections("nonfiction_texts.qmd")
    print_commentless_as_wip = True
    print_subtitle = True

rating_maps = {
    8: "❤️",
    9: "❤️❤️",
    10: "❤️‍🔥❤️‍🔥❤️‍🔥"
}

def flag_emoji(code):
    return ''.join(chr(127397 + ord(c)) for c in code.upper())

content = "<div class=\"books\">" # inside this is sortable
end_matters = ""

for i in df.index:
    row = df.loc[i]
    ranking = i + 1
    author = row["Author"]
    cover = row["CoverFile"]
    if pd.isna(cover):
        cover = "book_cover_generic.jpeg"
    title = row["Title"]
    subtitle = row["Subtitle"]
    year = int(row["Year"])
    rating = int(row["Rating"])
    rating_formatted = rating_maps[rating] if map_ratings else f"{rating}/10"
    genre = row["Genres"]
    tags = row["Tags"]
    country = row["Country"]
    country_name = row["CountryName"]
    comment = row["Comment"]

    if title not in external_comments and print_commentless_as_wip:
        end_matters += f"- {author} - {title}\n"
        continue

    if pd.isna(tags):
        tags = []
    else:
        tags = tags.split(",")

    book_classes = f'{{.book data-country="{country_name}" data-rating="{rating}" data-year="{year}" data-title="{title}" data-author="{author}"}}'
    if print_tags:
        all_tags = ""
        for tag in tags:
            all_tags += " ." + tag.encode("ascii", "ignore").decode().lower().strip().replace(" ", "_")
        book_classes = book_classes.replace(".book", ".book" + all_tags)
    content += f"## {author} - {title}{book_classes}\n"
    content += f"""<div class="album-container"><div class="album-image">![](../img/{cover})</div><div class="album-info"><p class="mb-0">"""
    if print_subtitle and subtitle:
        content += f"<strong>Subtitle</strong>: {subtitle}<br>"
    content += f"""<strong>Year:</strong> {year}<br>"""
    if print_country:
        content += f"<strong>Country:</strong> {flag_emoji(country)} {country_name}<br>"
    content += f"<strong>Genres:</strong> {genre}<br>"
    content += f"<strong>Verdict:</strong> {rating_formatted}<br>"
    content += "</p></div></div>"

    if print_tags:
        tag_string = """<div class="toggle-tags"><div class="d-flex flex-wrap gap-2 mb-2 mt-2">"""
        for tag in tags:
            tag_formatted = tag.encode("ascii", "ignore").decode().lower().strip().replace(" ", "_")
            tag_string += f"""<button id="{tag_formatted}" class="badge bg-primary rounded-pill border-0" onclick="toggleTag(this, '{tag_formatted}')">{tag}</button>"""
            #tag_string += f"""<span class="badge bg-primary rounded-pill">{tag.strip()}</span>"""
        tag_string += "</div></div>"
        content += tag_string
    content += "\n"

    if (pd.notna(comment) or title in external_comments) and print_comments:
        if title in external_comments:
            comment = external_comments[title]
            content += f"""\n{comment}\n\n"""
        else:
            content += f"""\n<span style="color:RoyalBlue">{comment}</span>\\\n\n"""

content += "</div>"

if end_matters != "":
    content += "\n\n## To be added\n\n"
    content += end_matters

with open(TEMPLATE_FILE, encoding="utf-8") as f:
    template = f.read()

final = template.replace("{{CONTENT}}", content)

with open(OUTPUT_QMD, "w", encoding="utf-8") as f:
    f.write(final)

print(f"Generated {OUTPUT_QMD}")

Generated books_nonfiction.qmd
